# Intelligent Agents: Reflex-Based Agents for the Vacuum-cleaner World

Student Name: Tenealle Sloan

I have used the following AI tools: AIMA Scholar

I understand that my submission needs to be my own work: T.S.

## Learning Outcomes

* Design and build a simulation environment that models sensor inputs, actuator effects, and performance measurement.
* Apply core AI concepts by implementing the agent function for a simple and model-based reflex agents that respond to environmental percepts.
* Practice how the environment and the agent function interact.
* Analyze agent performance through controlled experiments across different environment configurations.
* Graduate Students: Develop strategies for handling uncertainty and imperfect information in autonomous agent systems.

## Instructions

Total Points: Undergrads 100 + 5 bonus / Graduate students 110

Complete this notebook. Use the provided notebook cells and insert additional code and markdown cells as needed. Submit the completely rendered notebook as a HTML file.

### AI Use

Here are some guidelines that will make it easier for you:

* __Don't:__ Rely on AI auto completion. You will waste a lot of time trying to figure out how the suggested code relates to what we do in class. Turn off AI code completion (e.g., Copilot) in your IDE.
* __Don't:__ Do not submit code/text that you do not understand or have not checked to make sure that it is complete and correct.
* __Do:__ Use AI for debugging and letting it explain code and concepts from class.

### Using Visual Studio Code

If you use VS code then you can use `Export` (click on `...` in the menu bar) to save your notebook as a HTML file. Note that you have to run all blocks before so the HTML file contains your output.

### Using Google Colab

In Colab you need to save the notebook on GoogleDrive to work with it. For this you need to mount your google dive and change to the correct directory by uncommenting the following lines and running the code block.

In [ ]:
# from google.colab import drive
# import os
#
# drive.mount('/content/drive')
# os.chdir('/content/drive/My Drive/Colab Notebooks/')

Once you are done with the assignment and have run all code blocks using `Runtime/Run all`, you can convert the file on your GoogleDrive into HTML be uncommenting the following line and running the block.

In [ ]:
# !jupyter nbconvert --to html Copy\ of\ robot_vacuum.ipynb

You may have to fix the file location or the file name to match how it looks on your GoogleDrive. You can navigate in Colab to your GoogleDrive using the little folder symbol in the navigation bar to the left.

## Introduction

In this assignment you will implement a simulator environment for an automatic vacuum cleaner robot, a set of different reflex-based agent programs, and perform a comparison study for cleaning a single room. Focus on the __cleaning phase__ which starts when the robot is activated and ends when the last dirty square in the room has been cleaned. Someone else will take care of the agent program needed to navigate back to the charging station after the room is clean.

## PEAS description of the cleaning phase

__Performance Measure:__ Each action costs 1 energy unit. The performance is measured as the sum of the energy units used to clean the whole room.

__Environment:__ A room with $n \times n$ squares where $n = 5$. Dirt is randomly placed on each square with probability $p = 0.2$. For simplicity, you can assume that the agent knows the size and the layout of the room (i.e., it knows $n$). To start, the agent is placed on a random square.

__Actuators:__ The agent can clean the current square (action `suck`) or move to an adjacent square by going `north`, `east`, `south`, or `west`.

__Sensors:__ Four bumper sensors, one for north, east, south, and west; a dirt sensor reporting dirt in the current square.  


## The agent program for a simple randomized agent

The agent program is a function that gets sensor information (the current percepts) as the arguments. The arguments are:

* A dictionary with boolean entries for the for bumper sensors `north`, `east`, `west`, `south`. E.g., if the agent is on the north-west corner, `bumpers` will be `{"north" : True, "east" : False, "south" : False, "west" : True}`.
* The dirt sensor produces a boolean.

The agent returns the chosen action as a string.

Here is an example implementation for the agent program of a simple randomized agent:  

In [1]:
# make sure numpy is installed
%pip install -q numpy

In [2]:
import numpy as np

actions = ["north", "east", "west", "south", "suck"]

def simple_randomized_agent(bumpers, dirty):
    return np.random.choice(actions)

In [3]:
# define percepts (current location is NW corner and it is dirty)
bumpers = {"north" : True, "east" : False, "south" : False, "west" : True}
dirty = True

# call agent program function with percepts and it returns an action
simple_randomized_agent(bumpers, dirty)

np.str_('west')

__Note:__ This is not a rational intelligent agent. It ignores its sensors and may bump into a wall repeatedly or not clean a dirty square. You will be asked to implement rational agents below.

## Simple environment example

We implement a simple simulation environment that supplies the agent with its percepts.
The simple environment is infinite in size (bumpers are always `False`) and every square is always dirty, even if the agent cleans it. The environment function returns a different performance measure than the one specified in the PEAS description! Since the room is infinite and all squares are constantly dirty, the agent can never clean the whole room. Your implementation needs to implement the **correct performance measure.** The energy budget of the agent is specified as `max_steps`.

In [4]:
def simple_environment(agent_function, max_steps, verbose = True):
    num_cleaned = 0

    for i in range(max_steps):
        dirty = True
        bumpers = {"north" : False, "south" : False, "west" : False, "east" : False}

        action = agent_function(bumpers, dirty)
        if (verbose): print("step", i , "- action:", action)

        if (action == "suck"):
            num_cleaned = num_cleaned + 1

    return num_cleaned



Do one simulation run with a simple randomized agent that has enough energy for 20 steps.

In [5]:
simple_environment(simple_randomized_agent, max_steps = 20)

step 0 - action: suck
step 1 - action: north
step 2 - action: east
step 3 - action: west
step 4 - action: north
step 5 - action: east
step 6 - action: south
step 7 - action: south
step 8 - action: east
step 9 - action: north
step 10 - action: east
step 11 - action: east
step 12 - action: north
step 13 - action: west
step 14 - action: west
step 15 - action: west
step 16 - action: south
step 17 - action: suck
step 18 - action: east
step 19 - action: suck


3

# Tasks

## General [10 Points]

1. Make sure that you use the latest version of this notebook.
2. Your implementation can use libraries like math, numpy, scipy, but not libraries that implement intelligent agents or complete search algorithms. Try to keep the code simple! In this course, we want to learn about the algorithms and we often do not need to use object-oriented design.
3. You notebook needs to be formatted professionally.
    - Add additional markdown blocks for your description, comments in the code, add tables and use mathplotlib to produce charts where appropriate
    - Do not show debugging output or include an excessive amount of output.
    - Check that your submitted file is readable and contains all figures.
4. Document your code. Use comments in the code and add a discussion of how your implementation works and your design choices.


## Task 1: Implement a simulation environment [20 Points]

The simple environment above is not very realistic. Your environment simulator needs to follow the PEAS description from above. It needs to:

* Initialize the environment by storing the state of each square (clean/dirty) and making some dirty. ([Help with random numbers and arrays in Python](https://github.com/mhahsler/CS7320-AI/blob/master/HOWTOs/random_numbers_and_arrays.ipynb))
* Keep track of the agent's position.
* Call the agent function repeatedly and provide the agent function with the sensor inputs.  
* React to the agent's actions. E.g, by removing dirt from a square or moving the agent around unless there is a wall in the way.
* Keep track of the performance measure. That is, track the agent's actions until all dirty squares are clean and count the number of actions it takes the agent to complete the task.

The easiest implementation for the environment is to hold an 2-dimensional array to represent if squares are clean or dirty and to call the agent function in a loop until all squares are clean or a predefined number of steps have been reached (i.e., the robot runs out of energy).

The simulation environment should be a function like the `simple_environment()` and needs to work with the simple randomized agent program from above. **Use the same environment for all your agent implementations in the tasks below.**

*Note on debugging:* Debugging can be difficult. Here are a few options:

* Make sure your environment prints enough information when you use `verbose = True`.
* VSCode also provides a very good interactive Python debugger for notebooks. Read the [HOWTO on debugging](https://github.com/mhahsler/CS7320-AI/blob/master/HOWTOs/debugging_in_notebooks.ipynb) for more details.
* Another very useful debugging help is to implement a function that the environment can use to displays the room with dirt and the current position of the robot at every step. You can use simple characters for dirt and the robot location.  

In [8]:
import numpy as np

class VacuumEnvironment:
    """
    The purpose of this environment is to simulate the vacuum cleaner world.
    """

    def __init__(self, n=5, dirt_probability=0.2):
        """
        This initializes the environment with n: size of the room (nxn) and
        dirt probability: probability that a square is dirty (0-1).
        """
        self.n = n
        self.dirt_probability = dirt_probability

        self.grid = (np.random.random((n, n)) < dirt_probability).astype(int)

        self.agent_row = np.random.randint(0, n)
        self.agent_col = np.random.randint(0, n)

        self.energy_used = 0

    def get_percepts(self):
        """
        Returns current percepts for the agent, which are bumpers that return
        boolean values for each direction, indicating whether there is wall there
        or not. This also returns dirty, which is a boolean variable indicating
        whether that cell is dirty.
        """
        bumpers = {
            "north": self.agent_row == 0,
            "south": self.agent_row == self.n - 1,
            "west": self.agent_col == 0,
            "east": self.agent_col == self.n - 1
        }

        dirty = self.grid[self.agent_row, self.agent_col] == 1

        return bumpers, dirty

    def execute_action(self, action):
        """
        Executes an action and updates the environment using the action string
        that indicates which direction the action needs to be taken and guides
        the agent in each direction. Then, returns energy cost, which will
        always be 1 because the function is called.
        """
        self.energy_used += 1

        if action == "suck":
            self.grid[self.agent_row, self.agent_col] = 0

        elif action == "north" and self.agent_row > 0:
            self.agent_row -= 1

        elif action == "south" and self.agent_row < self.n - 1:
            self.agent_row += 1

        elif action == "west" and self.agent_col > 0:
            self.agent_col -= 1

        elif action == "east" and self.agent_col < self.n - 1:
            self.agent_col += 1

        return 1

    def is_clean(self):
        """
        Returns boolean variable indicating whether any dirt remains.
        """
        return np.sum(self.grid) == 0

    def display(self):
        """
        Prints the current state of the environment.
        """
        print(f"Energy used: {self.energy_used}")
        print(f"Agent at: ({self.agent_row}, {self.agent_col})")

        for row in range(self.n):
            for col in range(self.n):
                if row == self.agent_row and col == self.agent_col:
                    print("A", end=" ")
                elif self.grid[row, col] == 1:
                    print("*", end=" ")
                else:
                    print(".", end=" ")
            print()
        print()


def run_simulation(environment, agent_program, max_steps=1000):
    """
    Runs the vacuum cleaner simulation. Where environment: VacuumEnvironment
    instance, agent_program: function that takes (bumpers, dirty) and returns
    action, and max_steps: maximum steps to prevent infinite loops. This
    function returns engergy_used: total energy consumed.
    """
    steps = 0

    while not environment.is_clean() and steps < max_steps:
        bumpers, dirty = environment.get_percepts()

        action = agent_program(bumpers, dirty)

        environment.execute_action(action)

        steps += 1

    return environment.energy_used

Show that your environment works with the simple randomized agent from above.

In [9]:
np.random.seed(42)
env = VacuumEnvironment(n=5, dirt_probability=0.2)

print("Initial state:")
env.display()

env2 = VacuumEnvironment(n=5, dirt_probability=0.2)
energy = run_simulation(env2, simple_randomized_agent)
print(f"Total energy used: {energy}")
print(f"Room is clean: {env2.is_clean()}")

Initial state:
Energy used: 0
Agent at: (2, 3)
. . . . * 
* * . . . 
* . . A * 
* . . . . 
. * . . . 

Total energy used: 292
Room is clean: True


## Task 2:  Implement a simple reflex agent [10 Points]

The simple reflex agent randomly walks around but reacts to the bumper sensor by not bumping into the wall and to dirt with sucking. Implement the agent program as a function.

_Note:_ Agents cannot directly use variable in the environment. They only gets the percepts as the arguments to the agent function. Use the function signature for the `simple_randomized_agent` function above.

In [10]:
def simple_reflex_agent(bumpers, dirty):
    """
    A simple reflex agent that reacts to its current percepts without memory.
    If the current square is dirty, it cleans it. Otherwise, it moves to a
    random adjacent square that does not have a wall. This agent does not
    remember where it has been or maintain any internal state.
    """
    if dirty:
        return "suck"

    valid_moves = []
    if not bumpers["north"]:
        valid_moves.append("north")
    if not bumpers["south"]:
        valid_moves.append("south")
    if not bumpers["east"]:
        valid_moves.append("east")
    if not bumpers["west"]:
        valid_moves.append("west")

    return np.random.choice(valid_moves)

Show how the agent works with your environment.

In [12]:
np.random.seed(42)

env = VacuumEnvironment(n=5, dirt_probability=0.2)

print("Initial state:")
env.display()

energy = run_simulation(env, simple_reflex_agent)
print(f"Total energy used: {energy}")
print(f"Room is clean: {env.is_clean()}")
env.display()

Initial state:
Energy used: 0
Agent at: (2, 3)
. . . . * 
* * . . . 
* . . A * 
* . . . . 
. * . . . 

Total energy used: 116
Room is clean: True
Energy used: 116
Agent at: (0, 4)
. . . . A 
. . . . . 
. . . . . 
. . . . . 
. . . . . 



## Task 3: Implement a model-based reflex agent [20 Points]

Model-based agents use a state to keep track of what they have done and perceived so far. Your agent needs to find out where it is located and then keep track of its current location. You also need a set of rules based on the state and the percepts to make sure that the agent will clean the whole room. For example, the agent can move to a corner to determine its location and then it can navigate through the whole room and clean dirty squares.

Describe how you define the __agent state__ and how your agent works before implementing it. ([Help with implementing state information in Python](https://github.com/mhahsler/CS7320-AI/blob/master/HOWTOs/store_agent_state_information.ipynb))

#### State and Implementation Description

The purpose of the model-based reflex agent is to maintain an internal model of the environment to make more intelligent decisions than the simple reflex agent. This agent keeps track of which squares have been visited and cleaned by storing their positions in a set. The agent also tracks its current position in the grid and the room size dimensions.

The agent's decision logic works as follows: if the current square is dirty, the agent cleans it and marks that position as cleaned in its internal memory. After cleaning or when preparing to move, the agent marks its current position as cleaned. When choosing where to move next, the agent evaluates all adjacent squares that do not have walls. It preferentially selects unvisited adjacent squares over squares it has already visited. If all adjacent squares have been visited, the agent moves randomly to any valid direction.

This implementation ensures more systematic exploration of the room compared to pure random movement. By maintaining memory of cleaned squares, the agent reduces wasted energy from repeatedly visiting the same clean squares, resulting in better overall performance.



In [13]:
def model_based_reflex_agent(bumpers, dirty):

    if not hasattr(model_based_reflex_agent, 'cleaned'):
        model_based_reflex_agent.cleaned = set()
        model_based_reflex_agent.current_pos = (0, 0)

    if dirty:
        model_based_reflex_agent.cleaned.add(model_based_reflex_agent.current_pos)
        return "suck"

    model_based_reflex_agent.cleaned.add(model_based_reflex_agent.current_pos)

    row, col = model_based_reflex_agent.current_pos

    unvisited_moves = []
    valid_moves = []

    if not bumpers["north"]:
        next_pos = (row - 1, col)
        valid_moves.append("north")
        if next_pos not in model_based_reflex_agent.cleaned:
            unvisited_moves.append("north")

    if not bumpers["south"]:
        next_pos = (row + 1, col)
        valid_moves.append("south")
        if next_pos not in model_based_reflex_agent.cleaned:
            unvisited_moves.append("south")

    if not bumpers["east"]:
        next_pos = (row, col + 1)
        valid_moves.append("east")
        if next_pos not in model_based_reflex_agent.cleaned:
            unvisited_moves.append("east")

    if not bumpers["west"]:
        next_pos = (row, col - 1)
        valid_moves.append("west")
        if next_pos not in model_based_reflex_agent.cleaned:
            unvisited_moves.append("west")

    if unvisited_moves:
        action = np.random.choice(unvisited_moves)
    else:
        action = np.random.choice(valid_moves)

    if action == "north":
        model_based_reflex_agent.current_pos = (row - 1, col)
    elif action == "south":
        model_based_reflex_agent.current_pos = (row + 1, col)
    elif action == "east":
        model_based_reflex_agent.current_pos = (row, col + 1)
    elif action == "west":
        model_based_reflex_agent.current_pos = (row, col - 1)

    return action


def reset_model_based_agent():

    if hasattr(model_based_reflex_agent, 'cleaned'):
        del model_based_reflex_agent.cleaned
        del model_based_reflex_agent.current_pos

Show how the agent works with your environment.

In [14]:
np.random.seed(42)

env = VacuumEnvironment(n=5, dirt_probability=0.2)

print("Initial state:")
env.display()

reset_model_based_agent()
energy = run_simulation(env, model_based_reflex_agent)
print(f"Total energy used: {energy}")
print(f"Room is clean: {env.is_clean()}")

Initial state:
Energy used: 0
Agent at: (2, 3)
. . . . * 
* * . . . 
* . . A * 
* . . . . 
. * . . . 

Total energy used: 30
Room is clean: True


## Task 4: Simulation study [30 Points]

Compare the performance (the performance measure is defined in the PEAS description above) of the agents using  environments of different size. Do at least $5 \times 5$, $10 \times 10$ and
$100 \times 100$. Use 100 random runs for each. Present the results using tables and graphs. Discuss the differences between the agents.
([Help with charts and tables in Python](https://github.com/mhahsler/CS7320-AI/blob/master/HOWTOs/charts_and_tables.ipynb))

In [15]:
def run_experiments(agent_program, agent_name, sizes=[5, 10, 100], num_runs=100, dirt_prob=0.2):
    """
    Runs multiple simulation experiments for a given agent across different
    environment sizes. For each size, runs the specified number of trials and
    collects energy usage statistics. Returns a dictionary containing results
    for each environment size including mean, standard deviation, minimum and
    maximum energy used.
    """
    results = {}

    for size in sizes:
        print(f"Running {num_runs} trials for {agent_name} on {size}x{size} environment...")
        energy_values = []

        for run in range(num_runs):
            if agent_name == "Model-based Reflex Agent":
                reset_model_based_agent()

            env = VacuumEnvironment(n=size, dirt_probability=dirt_prob)
            energy = run_simulation(env, agent_program, max_steps=size*size*100)
            energy_values.append(energy)

        results[size] = {
            'mean': np.mean(energy_values),
            'std': np.std(energy_values),
            'min': np.min(energy_values),
            'max': np.max(energy_values),
            'all_values': energy_values
        }

        print(f"  {size}x{size}: Mean = {results[size]['mean']:.2f}, Std = {results[size]['std']:.2f}")

    return results

#### All Experiments

In [ ]:
randomized_results = run_experiments(simple_randomized_agent, "Randomized Agent")
print()

simple_reflex_results = run_experiments(simple_reflex_agent, "Simple Reflex Agent")
print()

model_based_results = run_experiments(model_based_reflex_agent, "Model-based Reflex Agent")
print()

Running 100 trials for Randomized Agent on 5x5 environment...
  5x5: Mean = 502.46, Std = 309.32
Running 100 trials for Randomized Agent on 10x10 environment...
  10x10: Mean = 2871.89, Std = 1154.64
Running 100 trials for Randomized Agent on 100x100 environment...


Fill out the following table with the average performance measure for 100 random runs (you may also create this table with code):

| Size     | Randomized Agent | Simple Reflex Agent | Model-based Reflex Agent |
|----------|------------------|---------------------|--------------------------|
| 5x5     | 502.46 | | |
| 10x10   | 2871.89 | | |
| 100x100 | | | |

Add charts to compare the performance of the different agents.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sizes = [5, 10, 100]
x_positions = np.arange(len(sizes))
width = 0.25

randomized_means = [randomized_results[s]['mean'] for s in sizes]
simple_reflex_means = [simple_reflex_results[s]['mean'] for s in sizes]
model_based_means = [model_based_results[s]['mean'] for s in sizes]

axes[0].bar(x_positions - width, randomized_means, width, label='Randomized Agent', alpha=0.8)
axes[0].bar(x_positions, simple_reflex_means, width, label='Simple Reflex Agent', alpha=0.8)
axes[0].bar(x_positions + width, model_based_means, width, label='Model-based Reflex Agent', alpha=0.8)

axes[0].set_xlabel('Environment Size')
axes[0].set_ylabel('Average Energy Used')
axes[0].set_title('Agent Performance Comparison Across Environment Sizes')
axes[0].set_xticks(x_positions)
axes[0].set_xticklabels([f'{s}x{s}' for s in sizes])
axes[0].legend()
axes[0].grid(axis='y', alpha=0.3)

for size in sizes:
    all_data = [
        randomized_results[size]['all_values'],
        simple_reflex_results[size]['all_values'],
        model_based_results[size]['all_values']
    ]

    bp = axes[1].boxplot(all_data, positions=[sizes.index(size)*4 + i for i in range(3)],
                         widths=0.6, patch_artist=True,
                         labels=['Random', 'Simple', 'Model'] if size == sizes[0] else ['', '', ''])

    colors = ['lightblue', 'lightgreen', 'lightcoral']
    for patch, color in zip(bp['boxes'], colors):
        patch.set_facecolor(color)

axes[1].set_xlabel('Environment Size and Agent Type')
axes[1].set_ylabel('Energy Used')
axes[1].set_title('Distribution of Energy Usage Across 100 Runs')
axes[1].set_xticks([1, 5, 9])
axes[1].set_xticklabels([f'{s}x{s}' for s in sizes])
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## Task 5: Robustness of the agent implementations [10 Points]

Describe how **your agent implementations** will perform

* if it is put into a rectangular room with unknown size,
* if the cleaning area can have an irregular shape (e.g., a hallway connecting two rooms), or
* if the room contains obstacles (i.e., squares that it cannot pass through and trigger the bumper sensors).
* if the dirt sensor is not perfect and gives 10% of the time a wrong reading (clean when it is dirty or dirty when it is clean).
* if the bumper sensor is not perfect and 10% of the time does not report a wall when there is one.

In [ ]:
# Answers goes here

## Advanced task: Imperfect Dirt Sensor

* __Graduate students__ need to complete this task [10 points]
* __Undergraduate students__ can attempt this as a bonus task [max +5 bonus points].

1. Change your simulation environment to run experiments for the following problem: The dirt sensor has a 10% chance of giving the wrong reading. Perform experiments to observe how this changes the performance of the three implementations. Your model-based reflex agent is likely not able to clean the whole room, so you need to measure performance differently as a tradeoff between energy cost and number of uncleaned squares.

2. Design an implement a solution for your model-based agent that will clean better. Show the improvement with experiments.

In [ ]:
# Your code and discussion goes here

## More Advanced Implementation (not for credit)

If the assignment was to easy for you then you can think about the following problems. These problems are challenging and not part of this assignment. We will learn implementation strategies and algorithms useful for these tasks during the rest of the semester.

* __Obstacles:__ Change your simulation environment to run experiments for the following problem: Add random obstacle squares that also trigger the bumper sensor. The agent does not know where the obstacles are. Perform experiments to observe how this changes the performance of the three implementations. Describe what would need to be done to perform better with obstacles. Add code if you can.

* __Agent for and environment with obstacles:__ Implement an agent for an environment where the agent does not know how large the environment is (we assume it is rectangular), where it starts or where the obstacles are. An option would be to always move to the closest unchecked/uncleaned square (note that this is actually depth-first search).

* __Utility-based agent:__ Change the environment for a $5 \times 5$ room, so each square has a fixed probability of getting dirty again. For the implementation, we give the environment a 2-dimensional array of probabilities. The utility of a state is defined as the number of currently clean squares in the room. Implement a utility-based agent that maximizes the expected utility over one full charge which lasts for 100000 time steps. To do this, the agent needs to learn the probabilities with which different squares get dirty again. This is very tricky!

In [ ]:
# Your ideas/code

&copy; 2025 [Michael Hahsler](http://michael.hahsler.net).
This work is openly licensed under [Creative Commons Attribution-ShareAlike 4.0 International (CC BY-SA 4.0) License](https://creativecommons.org/licenses/by-sa/4.0/)

![CC BY-SA 4.0](https://licensebuttons.net/l/by-sa/3.0/88x31.png)